In [1]:
# 6-26-2026

In [2]:
import xarray as xr
import numpy as np
import pandas as pd

In [3]:
ds_path = "../data/seasfire_pyromes_ecoregions.zarr"
domains_path = "../data/ecoregion_domains.zarr"

In [ ]:
# this section is for getting fire sparsity
# its calculated as (num fire cells)/(total cells) per domain

In [4]:
ds = xr.open_zarr(ds_path, consolidated=True)
domains = xr.open_zarr(domains_path, consolidated=True)

In [8]:
domain_arr = domains["domain_id"].values
ba = ds["gwis_ba"].values  # (time, lat, lon)

In [9]:
unique_domains = np.unique(domain_arr)
unique_domains = unique_domains[unique_domains != -1]

In [10]:
sparsity = {}
for domain in unique_domains:
    mask = (domain_arr == domain)
    domain_ba = ba[:, mask].ravel()  # all (lat, lon, time) entries for this domain
    total = len(domain_ba)
    fire = (domain_ba > 0).sum()
    sparsity[int(domain)] = fire / total

In [11]:
sparsity_df = pd.DataFrame.from_dict(sparsity, orient="index", columns=["fire_sparsity"])
sparsity_df.index.name = "domain_id"

In [ ]:
sparsity_df.head(15) # good, domains like 11 have high value

,fire_sparsity
domain_id,
0,0.081824
1,0.013560
2,0.032987
3,0.001419
4,0.007227
5,0.058533
6,0.003038
7,0.004830
8,0.019789


In [14]:
# this section is for land cover diversity, will use simpson diversity index formula

In [17]:
lccs_vars = ["lccs_class_1", "lccs_class_2", "lccs_class_3", "lccs_class_4", "lccs_class_6", "lccs_class_7"]

# build a (num_classes, num_domains) array of mean proportions
domain_props = np.zeros((len(lccs_vars), len(unique_domains)))

In [18]:
for i, var in enumerate(lccs_vars):
    print(f"loading {var}...")
    data = ds[var].values  # (time, lat, lon)
    for j, domain in enumerate(unique_domains):
        mask = (domain_arr == domain)
        domain_props[i, j] = np.nanmean(data[:, mask])
    del data

diversity = {}
for j, domain in enumerate(unique_domains):
    p = domain_props[:, j]
    p = p[p > 0]
    p = p / p.sum() # normalize jus in case
    diversity[int(domain)] = 1 - np.sum(p ** 2)

loading lccs_class_1...
loading lccs_class_2...
loading lccs_class_3...
loading lccs_class_4...
loading lccs_class_6...
loading lccs_class_7...


In [19]:
diversity_df = pd.DataFrame.from_dict(diversity, orient="index", columns=["land_cover_diversity"])
diversity_df.index.name = "domain_id"

In [21]:
diversity_df.head(10)

,land_cover_diversity
domain_id,
0,0.728328
1,0.218936
2,0.602900
3,0.365885
4,0.615066
5,0.560294
6,0.626834
7,0.655366
8,0.553258


In [ ]:
# for fire seasonality, according to the archibald paper its the num months where 80% of ba occurred
# for each domain, sum up the total amount of burned area per month, order by amount, 
#   keep adding top ba months until 80% of total ba is surpassed

In [23]:
print(ds.time)

<xarray.DataArray 'time' (time: 506)> Size: 4kB
array(['2011-01-01T00:00:00.000000000', '2011-01-09T00:00:00.000000000',
       '2011-01-17T00:00:00.000000000', ..., '2021-12-11T00:00:00.000000000',
       '2021-12-19T00:00:00.000000000', '2021-12-27T00:00:00.000000000'],
      shape=(506,), dtype='datetime64[ns]')
Coordinates:
  * time     (time) datetime64[ns] 4kB 2011-01-01 2011-01-09 ... 2021-12-27
Attributes:
    description:  Each datetime initializes the first date of the 8 days time...
    format:       YYYY-MM-DD
    type:         datetime


In [24]:
# extract month number (1 to 12) for the 506 timesteps
time_months = pd.DatetimeIndex(ds.time.values).month

print(time_months)
print(f"shape: {time_months.shape}")
print(f"unique months present: {np.unique(time_months)}")

Index([ 1,  1,  1,  1,  2,  2,  2,  2,  3,  3,
       ...
       10, 10, 11, 11, 11, 11, 12, 12, 12, 12],
      dtype='int32', length=506)
shape: (506,)
unique months present: [ 1  2  3  4  5  6  7  8  9 10 11 12]


In [25]:
ba = ds["gwis_ba"].values  # (time, lat, lon)

# build monthly ba totals per domain -- shape (12, num_domains)
monthly_ba = np.zeros((12, len(unique_domains)))

for j, domain in enumerate(unique_domains):
    mask = (domain_arr == domain)
    domain_ba = ba[:, mask]  # (time, num_cells_in_domain)
    for m in range(1, 13):
        month_mask = (time_months == m)
        monthly_ba[m - 1, j] = np.nansum(domain_ba[month_mask, :]) # index offset


In [ ]:
print(f"monthly_ba shape: {monthly_ba.shape}")
print(f"sample column (domain 0): {monthly_ba[:, 0]}")
# most fires in middle months

monthly_ba shape: (12, 50)
sample column (domain 0): [20309604.  21443028.  13534785.   4607522.5  4627252.  10562540.
 35834192.  81773024.  72868624.  22028944.  12388693.  13047752. ]


In [27]:
seasonality = {}

for j, domain in enumerate(unique_domains):
    col = monthly_ba[:, j]
    total = col.sum()
    if total == 0:
        seasonality[int(domain)] = np.nan
        continue
    sorted_col = np.sort(col)[::-1]  # descending
    cumsum = np.cumsum(sorted_col)
    n_months = np.searchsorted(cumsum, 0.8 * total) + 1  # +1 because searchsorted starts index 0
    seasonality[int(domain)] = n_months

In [28]:
print(seasonality)

{0: np.int64(6), 1: np.int64(7), 2: np.int64(7), 3: np.int64(3), 4: np.int64(5), 5: np.int64(4), 6: np.int64(4), 7: np.int64(3), 8: np.int64(5), 9: np.int64(2), 10: np.int64(5), 11: np.int64(4), 12: np.int64(5), 13: np.int64(5), 14: np.int64(4), 15: np.int64(5), 16: np.int64(5), 17: np.int64(2), 18: np.int64(6), 19: np.int64(6), 20: np.int64(4), 21: np.int64(4), 22: np.int64(5), 23: np.int64(4), 24: np.int64(3), 25: np.int64(4), 26: np.int64(5), 27: np.int64(7), 28: np.int64(5), 29: np.int64(6), 30: np.int64(3), 31: np.int64(6), 32: np.int64(6), 33: np.int64(6), 34: np.int64(6), 35: np.int64(3), 36: np.int64(3), 37: np.int64(5), 38: np.int64(6), 39: np.int64(9), 40: np.int64(2), 41: np.int64(1), 42: np.int64(3), 43: np.int64(4), 44: np.int64(1), 45: np.int64(4), 46: np.int64(2), 47: np.int64(5), 48: np.int64(3), 49: np.int64(6)}


In [29]:
seasonality_df = pd.DataFrame.from_dict(seasonality, orient="index", columns=["fire_season_length"])
seasonality_df.index.name = "domain_id"

In [31]:
seasonality_df.head(10)

,fire_season_length
domain_id,
0,6
1,7
2,7
3,3
4,5
5,4
6,4
7,3
8,5


In [32]:
# now need to join all df's + simple stats df

In [33]:
simple_stats = pd.read_csv("simple_domain_stats.csv")

In [34]:
simple_stats.head()

,domain_id,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,...,swvl1_p90,swvl4_mean,swvl4_std,swvl4_p90,gwis_ba_mean,gwis_ba_std,gwis_ba_p90,cams_frpfire_mean,cams_frpfire_std,cams_frpfire_p90
0,0,11.229313,5.895861,19.746270,2.516127,1.052111,3.968697,302.43317,4.790887,307.46002,...,0.477633,0.349587,0.107862,0.479085,547.38410,1932.2546,1112.03370,0.067789,0.469590,0.097816
1,1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.80650,4.735351,305.51370,...,0.490699,0.428787,0.055336,0.499585,338.66165,1157.0903,705.89197,0.063071,0.392797,0.108795
2,2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.89624,6.380806,311.61792,...,0.201105,0.156024,0.070416,0.257699,3360.49340,8181.0723,8625.16300,0.162555,0.925012,0.219152
3,3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.08792,10.954839,299.41890,...,0.399886,0.325719,0.103028,0.411259,328.76685,739.6544,901.44635,0.041860,0.274789,0.000000
4,4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.42078,16.251247,293.80110,...,0.426583,0.365052,0.094277,0.447848,669.05206,2307.7700,1315.65490,0.185203,1.571907,0.090595


In [35]:
simple_stats.shape

(50, 31)

In [36]:
descriptor_df = simple_stats.set_index("domain_id").join([sparsity_df, diversity_df, seasonality_df])

In [37]:
descriptor_df.head()

,vpd_mean,vpd_std,vpd_p90,ws10_mean,ws10_std,ws10_p90,t2m_max_mean,t2m_max_std,t2m_max_p90,ndvi_mean,...,swvl4_p90,gwis_ba_mean,gwis_ba_std,gwis_ba_p90,cams_frpfire_mean,cams_frpfire_std,cams_frpfire_p90,fire_sparsity,land_cover_diversity,fire_season_length
domain_id,,,,,,,,,,,,,,,,,,,,,
0,11.229313,5.895861,19.746270,2.516127,1.052111,3.968697,302.43317,4.790887,307.46002,0.641714,...,0.479085,547.38410,1932.2546,1112.03370,0.067789,0.469590,0.097816,0.081824,0.728328,6
1,6.742952,3.144529,10.522034,1.406203,0.511182,2.132592,301.80650,4.735351,305.51370,0.796118,...,0.499585,338.66165,1157.0903,705.89197,0.063071,0.392797,0.108795,0.013560,0.218936,7
2,21.599873,10.565010,36.654785,4.066474,0.951055,5.291819,303.89624,6.380806,311.61792,0.242577,...,0.257699,3360.49340,8181.0723,8625.16300,0.162555,0.925012,0.219152,0.032987,0.602900,7
3,4.211150,2.716757,7.958644,3.166607,1.505368,5.099028,286.08792,10.954839,299.41890,0.550528,...,0.411259,328.76685,739.6544,901.44635,0.041860,0.274789,0.000000,0.001419,0.365885,3
4,2.564834,3.033652,6.938553,3.300411,1.380530,5.114551,273.42078,16.251247,293.80110,0.315671,...,0.447848,669.05206,2307.7700,1315.65490,0.185203,1.571907,0.090595,0.007227,0.615066,5


In [38]:
# nice

In [40]:
descriptor_df.to_csv("domain_descriptions.csv")